In [8]:
import pandas as pd
import numpy as np
import sqlite3

# 1. Cargar el CSV expandido (Donde agregás manualmente tus casos de control externos)
df_entrenamiento = pd.read_csv('../data/adn_boca_expandido.csv')

# 2. Calcular las Features derivadas para el modelo
df_entrenamiento["goles_por_partido"] = df_entrenamiento["goles"] / df_entrenamiento["partidos"].replace(0, np.nan)
df_entrenamiento["asist_por_partido"] = df_entrenamiento["asistencias"] / df_entrenamiento["partidos"].replace(0, np.nan)
df_entrenamiento["rendimiento"]       = df_entrenamiento["rating"] * (df_entrenamiento["partidos"] / 38)
df_entrenamiento["participacion_gol"] = (df_entrenamiento["goles"] + df_entrenamiento["asistencias"]) / df_entrenamiento["partidos"].replace(0, np.nan)
df_entrenamiento["experiencia"]       = df_entrenamiento["temporada"] - (2024 - df_entrenamiento["edad"])
df_entrenamiento["pases_norm"]        = df_entrenamiento["pases_precisos"] / 90
df_entrenamiento["perfil_ofensivo"]   = (df_entrenamiento["posicion"].isin(["Attacker", "Midfielder"])).astype(int)

df_entrenamiento.fillna(0, inplace=True)

# Guardar copia local de features
df_entrenamiento.to_csv("../data/adn_boca_features.csv", index=False)

# 3. Guardar en SQLite de forma limpia (REPLACE evita la duplicación)
conexion = sqlite3.connect('../data/boca_juniors.db')

# Al usar 'replace', Pandas borra la tabla vieja y crea la nueva con los datos frescos
df_entrenamiento.to_sql('jugadores_entrenamiento', conexion, if_exists='replace', index=False)

conexion.close()
print(f"✅ Tabla 'jugadores_entrenamiento' guardada con éxito. Total: {len(df_entrenamiento)} registros.")

✅ Tabla 'jugadores_entrenamiento' guardada con éxito. Total: 193 registros.


In [9]:
import requests
import sqlite3
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv("../.gitignore\.env")
api_key = os.getenv("API_KEY")

headers = {'x-apisports-key': api_key}

def safe_int(value): return int(value) if value is not None else 0
def safe_float(value): return float(value) if value is not None else 0.0

# ==========================================
# PASO 1: OBTENER LOS ENLACES DE LOS EQUIPOS
# ==========================================
print("📋 Obteniendo la lista de equipos de la Liga Profesional (128)...")
url_equipos = "https://v3.football.api-sports.io/teams"
params_equipos = {
    'league': '128',
    'season': '2024'
}

response_equipos = requests.get(url_equipos, headers=headers, params=params_equipos)
equipos_data = response_equipos.json().get('response', [])

lista_equipos = []
for item in equipos_data:
    id_team = item['team']['id']
    nombre_team = item['team']['name']
    
    # Salteamos a Boca (451) porque ya lo tenés en tu dataset de entrenamiento
    if id_team == 451:
        continue
    
    lista_equipos.append({'id': id_team, 'name': nombre_team})

print(f"✅ Se encontraron {len(lista_equipos)} equipos rivales para escautear.")

# ==========================================
# PASO 2: BAJAR JUGADORES EQUIPO POR EQUIPO
# ==========================================
print("\n📊 Descargando candidatos del mercado de pases...")
url_jugadores = "https://v3.football.api-sports.io/players"
todos_los_candidatos = []

# Recorremos cada equipo encontrado
for equipo in lista_equipos:
    print(f"  -> Descargando jugadores de: {equipo['name']} (ID: {equipo['id']})...")
    pagina = 1
    
    while True:
        params_jugadores = {
            'team': str(equipo['id']),
            'season': '2024',
            'league': '128',
            'page': pagina
        }
        
        response = requests.get(url_jugadores, headers=headers, params=params_jugadores)
        data = response.json()
        jugadores_pagina = data.get('response', [])
        
        if not jugadores_pagina:
            break
            
        todos_los_candidatos.extend(jugadores_pagina)
        
        total_paginas = data.get('paging', {}).get('total', 1)
        if pagina >= total_paginas:
            break
        pagina += 1

print(f"\n✅ Total de registros crudos descargados del mercado: {len(todos_los_candidatos)}")

# ==========================================
# PASO 3: PROCESAR E INSERTAR EN SQLITE
# ==========================================
if todos_los_candidatos:
    lista_procesada = []
    for item in todos_los_candidatos:
        p = item['player']
        stats = item.get('statistics', [])
        if not stats: continue
        
        s = stats[0]
        
        lista_procesada.append({
            "nombre": p.get('name', 'Desconocido'),
            "club_actual": s.get('team', {}).get('name', 'Otros'),
            "posicion": s['games'].get('position') or 'N/A',
            "edad": safe_int(p.get('age')),
            "temporada": 2024,
            "partidos": safe_int(s['games'].get('appearences')),
            "goles": safe_int(s['goals'].get('total')),
            "asistencias": safe_int(s['goals'].get('assists')),
            "pases_precisos": safe_float(s['passes'].get('accuracy')),
            "rating": safe_float(s['games'].get('rating'))
        })
    
    df_mercado = pd.DataFrame(lista_procesada)
    
    # Guardar en la tabla limpia 'candidatos_mercado'
    conexion = sqlite3.connect('../data/boca_juniors.db')
    df_mercado.to_sql('candidatos_mercado', conexion, if_exists='replace', index=False)
    conexion.close()
    
    print(f"🚀 ¡Tabla 'candidatos_mercado' guardada en SQLite con {len(df_mercado)} jugadores!")
else:
    print("⚠️ No se pudo procesar ningún jugador.")

📋 Obteniendo la lista de equipos de la Liga Profesional (128)...
✅ Se encontraron 27 equipos rivales para escautear.

📊 Descargando candidatos del mercado de pases...
  -> Descargando jugadores de: Gimnasia L.P. (ID: 434)...
  -> Descargando jugadores de: River Plate (ID: 435)...
  -> Descargando jugadores de: Racing Club (ID: 436)...
  -> Descargando jugadores de: Rosario Central (ID: 437)...
  -> Descargando jugadores de: Velez Sarsfield (ID: 438)...
  -> Descargando jugadores de: Godoy Cruz (ID: 439)...
  -> Descargando jugadores de: Belgrano Cordoba (ID: 440)...
  -> Descargando jugadores de: Union Santa Fe (ID: 441)...
  -> Descargando jugadores de: Defensa Y Justicia (ID: 442)...
  -> Descargando jugadores de: Huracan (ID: 445)...
  -> Descargando jugadores de: Lanus (ID: 446)...
  -> Descargando jugadores de: Banfield (ID: 449)...
  -> Descargando jugadores de: Estudiantes L.P. (ID: 450)...
  -> Descargando jugadores de: Tigre (ID: 452)...
  -> Descargando jugadores de: Independ

In [10]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect('../data/boca_juniors.db')

# Agrupamos para ver cuántos tenés de cada uno
df_balance = pd.read_sql("""
    SELECT etiqueta, COUNT(*) as total_jugadores, AVG(rating) as rating_promedio 
    FROM jugadores_entrenamiento 
    GROUP BY etiqueta
""", conexion)

print("📊 Balance actual de tu dataset de entrenamiento:")
print(df_balance)

print("\n🔥 Últimos 5 jugadores etiquetados como ADN Boca (1):")
df_top_adn = pd.read_sql("""
    SELECT nombre, temporada, posicion, rating 
    FROM jugadores_entrenamiento 
    WHERE etiqueta = 1 
    LIMIT 5
""", conexion)
print(df_top_adn)

conexion.close()

📊 Balance actual de tu dataset de entrenamiento:
   etiqueta  total_jugadores  rating_promedio
0         0               82         6.687805
1         1              111         7.443243

🔥 Últimos 5 jugadores etiquetados como ADN Boca (1):
            nombre  temporada    posicion  rating
0     Carlos Tevez       2018    Attacker     7.8
1     Carlos Tevez       2019    Attacker     7.5
2  Dario Benedetto       2018    Attacker     7.9
3  Dario Benedetto       2019    Attacker     7.7
4    Fernando Gago       2018  Midfielder     7.6
